# Change Control, Collaboration, And CRDT Workflows

Examples for intents, named views, signed collaboration objects, gate checks, reusable resolutions, redactions, transports, actors, CRDT sync, and entity merges.

Run this cell from the repository root after `npm ci` and `npm run build`. The stored output below was regenerated by `npm run notebooks:build`.

## Intent Policy And Main Projection

**Use when:** Use this flow when proposed file changes should stay pending until signed maintainer action merges them.

The next cell is the executable example.

In [1]:
import { mkdtempSync, writeFileSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';
import { EpochRepository } from 'epoch';

const root = mkdtempSync(join(tmpdir(), 'epoch-intent-'));
const repo = EpochRepository.create(root, { author: 'alice' });
writeFileSync(join(root, 'change.txt'), 'draft\n');
const intent = repo.intentFile('change.txt', 'text/plain', 'alice', { title: 'Draft change', labels: ['docs'] });
const pending = repo.policy();
repo.comment('Looks good', intent.id, 'bob');
repo.mergeIntent(intent.id, 'maintainer', { title: 'Accept docs' });
const merged = repo.policy({ mergesRequired: 1 });

console.log(JSON.stringify({
  intentType: intent.type,
  pending: pending.pending.length,
  merged: merged.merged.length,
  mainPatches: repo.mainPatches({ mergesRequired: 1 }).map((patch) => patch.path),
  verifyProblems: repo.verify().length,
}, null, 2));

{
  "intentType": "intent",
  "pending": 1,
  "merged": 1,
  "mainPatches": [
    "change.txt"
  ],
  "verifyProblems": 0
}


**How to read the output:** The pending intent becomes part of the main projection only after the merge event satisfies policy.

## Named Views And Promotion

**Use when:** Use this flow when a branch-like view should isolate local records until promotion.

The next cell is the executable example.

In [2]:
import { mkdtempSync, writeFileSync, readFileSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';
import { EpochRepository } from 'epoch';

const root = mkdtempSync(join(tmpdir(), 'epoch-view-'));
const repo = EpochRepository.create(root, { author: 'alice' });
writeFileSync(join(root, 'policy.txt'), 'base\n');
repo.recordFile('policy.txt', 'text/plain');
repo.createView('feature/policy-update', { type: 'all' }, undefined, { owner: 'alice' });
repo.checkoutView('feature/policy-update');
writeFileSync(join(root, 'policy.txt'), 'feature\n');
repo.recordFile('policy.txt', 'text/plain');
const mainBefore = repo.computeViewState('main');
const feature = repo.computeViewState('feature/policy-update');
repo.promoteToView('feature/policy-update', 'main');
const mainAfter = repo.computeViewState('main');
repo.checkoutView('main');

console.log(JSON.stringify({
  featureOwner: repo.listViews().find((view) => view.name === 'feature/policy-update')?.metadata?.owner,
  mainBeforeIntents: mainBefore.intentIds.length,
  featureIntents: feature.intentIds.length,
  mainAfterIntents: mainAfter.intentIds.length,
  checkedOutPolicy: readFileSync(join(root, 'policy.txt'), 'utf8').trim(),
  verifyProblems: repo.verify().length,
}, null, 2));

{
  "featureOwner": "alice",
  "mainBeforeIntents": 1,
  "featureIntents": 2,
  "mainAfterIntents": 2,
  "checkedOutPolicy": "feature",
  "verifyProblems": 0
}


**How to read the output:** The feature view can carry more intent records than main, and promotion makes the feature content visible through main checkout.

## Collaboration, Gates, Redactions, Resolutions, And Bundle Transport

**Use when:** Use this flow when signed issues, reviews, CI, command history, secret cleanup, and offline handoff need one audit log.

The next cell is the executable example.

In [3]:
import { mkdtempSync, writeFileSync, existsSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';
import { EpochRepository, BundleEpochTransport } from 'epoch';

const root = mkdtempSync(join(tmpdir(), 'epoch-notebook-collab-'));
const repository = EpochRepository.create(root, { author: 'alice' });
writeFileSync(join(root, 'policy.txt'), 'signed gates\n');
const record = repository.recordFile('policy.txt', 'text/plain');
const issue = repository.createIssue('Track release gates', 'Use signed review and CI gates', 'alice');
const intent = repository.intentFile('policy.txt', 'text/plain', 'alice', { title: 'Policy update' });
repository.reviewIntent(intent.id, 'approved', 'Deterministic and audited', 'bob');
repository.recordCI('unit', 'passed', intent.id, 'ci-bot');
repository.appendOperation('notebook:collaboration', 'succeeded', { issueId: issue.id });
repository.recordConflictResolution({
  path: 'config.json',
  entityType: 'application/json',
  base: { flag: 0 },
  left: { flag: 1 },
  right: { flag: 2 },
  resolved: { flag: 3 },
});
const reused = repository.mergeEntity('config.json', 'application/json', { flag: 0 }, { flag: 1 }, { flag: 2 });
const gate = repository.gateStatus(intent.id, { requiredReviewState: 'approved', requiredCi: ['unit'] });
const blobHash = record.payload.blob_sha256;
const redactionPlan = repository.planRedaction(blobHash);
repository.redactBlob(blobHash, 'demo secret cleanup', 'security');

const peerRoot = mkdtempSync(join(tmpdir(), 'epoch-notebook-collab-peer-'));
const peer = EpochRepository.create(peerRoot, { author: 'peer' });
const bundlePath = join(root, 'sync.bundle');
BundleEpochTransport.write(bundlePath, repository.exportToMemoryTransport());
const sync = peer.syncWithTransport(BundleEpochTransport.read(bundlePath));
const collaboration = repository.collaboration();

console.log(JSON.stringify({
  collaborationSummary: {
    issues: collaboration.issues.length,
    reviewStates: collaboration.reviews.map((review) => review.state),
  },
  gate,
  operationCommands: repository.operations().map((operation) => operation.command),
  reusedResolution: reused,
  redactionPlan: {
    localBlobPresent: redactionPlan.localBlobPresent,
    affectedEvents: redactionPlan.eventIds.length,
    alreadyRedacted: redactionPlan.alreadyRedacted,
  },
  redactions: repository.redactions().map((redaction) => redaction.reason),
  bundleWritten: existsSync(bundlePath),
  syncedEvents: sync.eventsCopied,
  peerVerifyProblems: peer.verify().length,
}, null, 2));

{
  "collaborationSummary": {
    "issues": 1,
    "reviewStates": [
      "approved"
    ]
  },
  "gate": {
    "passed": true,
    "blockers": []
  },
  "operationCommands": [
    "notebook:collaboration"
  ],
  "reusedResolution": {
    "flag": 3
  },
  "redactionPlan": {
    "localBlobPresent": true,
    "affectedEvents": 2,
    "alreadyRedacted": false
  },
  "redactions": [
    "demo secret cleanup"
  ],
  "bundleWritten": true,
  "syncedEvents": 8,
  "peerVerifyProblems": 0
}


**How to read the output:** The result confirms the gate passes, the exact-match conflict resolution is reused, redaction is recorded, and the bundle transport can seed a verified peer.

## Actors, CRDT Operations, Sync, And Entity Merges

**Use when:** Use this flow when concurrent agents or users need serialized writes and convergent shared state.

The next cell is the executable example.

In [4]:
import { mkdtempSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';
import { EpochActorSystem, EpochRepository, CRDTRegistry, EntityRegistry, EntityType } from 'epoch';

const root = mkdtempSync(join(tmpdir(), 'epoch-notebook-actors-'));
const actors = new EpochActorSystem(root);
await actors.init('alice');
await Promise.all([
  actors.user('alice').appendCRDTOperation({ kind: 'map-set', entity: 'tasks', key: 'design', value: { status: 'draft' } }),
  actors.user('bob').appendCRDTOperation({ kind: 'map-set', entity: 'tasks', key: 'tests', value: { status: 'green' } }),
]);
const tasks = await actors.materialize('tasks');
const peerRoot = mkdtempSync(join(tmpdir(), 'epoch-notebook-peer-'));
const peer = EpochRepository.create(peerRoot, { author: 'peer' });
const sync = peer.syncFrom(root);
actors.stop();

const registry = CRDTRegistry.defaults();
const text = registry.merge(EntityType.plainText, 'alpha\nomega\n', 'alpha\nleft\nomega\n', 'alpha\nright\nomega\n');
const json = registry.merge(EntityType.json, { feature: false }, { feature: true }, { feature: false, docs: true });
const csv = EntityRegistry.defaults().merge(
  EntityType.csv,
  'id,status\n1,old\n',
  'id,status\n1,old\n2,left\n',
  'id,status\n1,new\n',
);

console.log(JSON.stringify({
  taskKeys: Object.keys(tasks).sort(),
  taskStatuses: Object.fromEntries(Object.entries(tasks).map(([key, value]) => [key, value.status])),
  syncedEvents: sync.eventsCopied,
  peerMaterialized: peer.materialize('tasks'),
  mergedTextLines: text.trim().split('\n'),
  mergedJson: json,
  mergedCsv: csv.trim().split('\n'),
  peerVerifyProblems: peer.verify().length,
}, null, 2));

{
  "taskKeys": [
    "design",
    "tests"
  ],
  "taskStatuses": {
    "design": "draft",
    "tests": "green"
  },
  "syncedEvents": 2,
  "peerMaterialized": {
    "design": {
      "status": "draft"
    },
    "tests": {
      "status": "green"
    }
  },
  "mergedTextLines": [
    "alpha",
    "left",
    "right",
    "omega"
  ],
  "mergedJson": {
    "docs": true,
    "feature": true
  },
  "mergedCsv": [
    "id,status",
    "1,new",
    "2,left"
  ],
  "peerVerifyProblems": 0
}


**How to read the output:** The actor facade serializes repository commands, CRDT replay converges on the peer, and built-in entity adapters merge text, JSON, and row-keyed CSV.